In [1]:
import numpy as np
import pandas as pd

In [2]:
area = pd.read_excel('./area/Flächennutzung.xls')
urban = pd.read_excel('./area/Siedlungsstruktur.xlsx', sheet_name='Planungsraum')
quality = pd.read_csv('./area/Wohnlage.csv', sep=';')
names = pd.read_excel('./names.xls', sheet_name='combined')

### Clean up Area

In [3]:
area['Nummer Planungsraum'] = area['Nummer Planungsraum'].apply('{:0>8}'.format)
area.drop('überwiegende Nutzung', axis=1, inplace=True)
area.replace('-', 0, inplace=True)

area['LOR'] = area['Nummer Planungsraum'].apply(lambda x: x[:6])

area.columns = ['PR', 'area', 'block_area', 'street_area', 'living_area', 'mixed_area', 
                'central_area', 'industrial_area', 'special_area', 'disposal_area',
                'datscha_area', 'traffic_area', 'forest_area', 'greens_area', 'farm_area',
                'park_area', 'platz_area', 'cemetery_area', 'garden_area', 'fallow_no_veg',
                'fallow_veg', 'fallow_mixed', 'nursery_area', 'water_area', 'LOR']

area['leisure_area'] = area['park_area'] + area['cemetery_area']\
                     + area['forest_area'] + area['greens_area']

# area in square km
area['area_sqkm'] = area['area'] / 1_000_000

#### Planungraum

In [4]:
area_pr = area[['PR', 'area', 'area_sqkm', 'living_area', 'leisure_area']].copy()

# percentage of park area per lor
area_pr['leisure_part'] = area_pr['leisure_area'] / area['area']

#### Bezirksregion

In [5]:
area_lor = area[['LOR', 'area', 'area_sqkm', 'living_area', 'leisure_area']].groupby('LOR').sum()

# percentage of park area per lor
area_lor['leisure_part'] = area_lor['leisure_area'] / area_lor['area']

### Clean up urbanization

In [6]:
urban['Nummer'] = urban['Nummer'].apply('{:0>8}'.format)
urban['LOR'] = urban['Nummer'].apply(lambda x: x[:6])
urban.replace('-', 0, inplace=True)

urban_final = urban[['Nummer', 'LOR', 'Wohnungsanzahl', 'm2', 'Einwohner',
                    '"Verdichtete Blockrandbebauung" Wohnungsanzahl', '"Großsiedlungen der 60er-80er Jahre" Wohnungsanzahl',
                    '"Einfamilienhausgebiet" Wohnungsanzahl',
                    '"Verdichtete Blockrandbebauung" in m2', '"Großsiedlungen der 60er-80er Jahre" in m2',
                    '"Einfamilienhausgebiete" in m2',
                    '"Verdichtete Blockrandbebauung" Einwohner', '"Großsiedlungen der 60er-80er Jahre" Einwohner',
                    '"Einfamilienhausgebiet" Einwohner']]

urban_final.columns = ['PR', 'LOR', 'flats', 'sqm', 'pop',
                       'altbau_flats', 'platte_flats', 'one_fam_flats',
                       'altbau_sqm', 'platte_sqm', 'one_fam_sqm',
                       'altbau_pop', 'platte_pop', 'one_fam_pop']

#### Planungsraum

In [7]:
apply_dict = {
    'altbau_flats': 'flats', 'platte_flats': 'flats', 'one_fam_flats': 'flats',
    'altbau_sqm': 'sqm', 'platte_sqm': 'sqm', 'one_fam_sqm': 'sqm',
    'altbau_pop': 'pop', 'platte_pop': 'pop', 'one_fam_pop': 'pop'
}

In [8]:
urban_pr = urban_final.copy()

for col, total in apply_dict.items():
    urban_pr[col] = urban_pr[col] / urban_pr[total]
    
urban_pr = urban_pr[['PR', 'altbau_flats', 'platte_flats','one_fam_flats',
                     'altbau_sqm', 'platte_sqm', 'one_fam_sqm',
                     'altbau_pop', 'platte_pop', 'one_fam_pop']]

#### Bezirksregion

In [9]:
urban_lor = urban_final.groupby('LOR', as_index=False).sum()

for col, total in apply_dict.items():
    urban_lor[col] = urban_lor[col] / urban_lor[total]
    
urban_lor = urban_lor[['LOR', 'altbau_flats', 'platte_flats','one_fam_flats',
                     'altbau_sqm', 'platte_sqm', 'one_fam_sqm',
                     'altbau_pop', 'platte_pop', 'one_fam_pop']]

### Clean up Quality

In [10]:
quality['PR'] = quality['RAUMID'].apply('{:0>8}'.format)
quality['LOR'] = quality['PR'].apply(lambda x: x[:6])

quality['total'] = quality[['WLEINFoL','WLEINFmL', 'WLMIToL', 'WLMITmL', 'WLGUToL', 'WLGUTmL', 'WLNZORD']].sum(axis=1)
quality['lower_quality'] = quality[['WLEINFoL','WLEINFmL']].sum(axis=1)
quality['average_quality'] = quality[['WLMIToL', 'WLMITmL']].sum(axis=1)
quality['upper_quality'] = quality[['WLGUToL', 'WLGUTmL']].sum(axis=1)

#### Planungsregion

In [11]:
quality_pr = quality.copy()

quality_pr['lower_quality'] = quality_pr['lower_quality'] / quality_pr['total']
quality_pr['average_quality'] = quality_pr['average_quality'] / quality_pr['total']
quality_pr['upper_quality'] = quality_pr['upper_quality'] / quality_pr['total']

quality_pr = quality_pr[['PR', 'lower_quality', 'average_quality', 'upper_quality']]

#### Bezirksregion

In [12]:
quality_lor = quality.groupby('LOR', as_index=False).sum()

quality_lor['lower_quality'] = quality_lor['lower_quality'] / quality_lor['total']
quality_lor['average_quality'] = quality_lor['average_quality'] / quality_lor['total']
quality_lor['upper_quality'] = quality_lor['upper_quality'] / quality_lor['total']

quality_lor = quality_lor[['LOR', 'lower_quality', 'average_quality', 'upper_quality']]

### Combine and add names

In [13]:
for col in ['Bezirk', 'Prognoseraum', 'Bezirksregion', 'Planungsraum']:
    names[col] = names[col].apply('{:0>2}'.format)

names['LOR'] = names['Bezirk'] + names['Prognoseraum'] + names['Bezirksregion']
names['PR'] = names['LOR'] + names['Planungsraum']

#### Planungsraum

In [14]:
pr_info = area_pr.merge(urban_pr, on='PR', how='outer').merge(quality_pr, on='PR', how='outer').merge(names, on='PR')

pr_info.fillna(0, inplace=True)

#### Bezirksregion

In [15]:
lor_info = area_lor.merge(urban_lor, on='LOR', how='outer') \
                   .merge(quality_lor, on='LOR', how='outer') \
                   .merge(names.drop(['Planungsraum', 'PR', 'Kiezname'], axis=1).drop_duplicates(), on='LOR', how='inner')

lor_info.fillna(0, inplace=True)

In [16]:
import joblib

joblib.dump(pr_info, './area_pr.csv')
joblib.dump(lor_info, './area_lor.csv')

['./area_lor.csv']

In [17]:
pr_info

,PR,area,area_sqkm,living_area,leisure_area,leisure_part,altbau_flats,platte_flats,one_fam_flats,altbau_sqm,...,upper_quality,Bezirk,Name,Prognoseraum,Progenoseraumname,Bezirksregion,Bezirksname,Planungsraum,Kiezname,LOR
0,00000000,890959877.0,890.959877,233329442.0,221881541.0,0.249037,0.000000,0.000000,0.000000,0.000000,...,0.193038,00,Berlin,00,Berlin,00,Berlin,00,Berlin,000000
1,01011101,366134.0,0.366134,32736.0,0.0,0.000000,0.442763,0.453289,0.000000,0.445021,...,0.392743,01,Mitte,01,Zentrum,11,Tiergarten Süd,01,Stülerstraße,010111
2,01011102,3008100.0,3.008100,0.0,1936147.0,0.643644,0.000000,0.000000,0.000000,0.000000,...,0.101064,01,Mitte,01,Zentrum,11,Tiergarten Süd,02,Großer Tiergarten,010111
3,01011103,523395.0,0.523395,0.0,7521.0,0.014370,1.000000,0.000000,0.000000,1.000000,...,0.000000,01,Mitte,01,Zentrum,11,Tiergarten Süd,03,Lützowstraße,010111
4,01011104,339411.0,0.339411,48123.0,27874.0,0.082125,0.628300,0.371700,0.000000,0.772605,...,0.000000,01,Mitte,01,Zentrum,11,Tiergarten Süd,04,Körnerstraße,010111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,12302211,1129447.0,1.129447,464894.0,60491.0,0.053558,0.000000,0.000000,0.513111,0.000000,...,0.000000,12,Reinickendorf,30,Waidmannslust,22,MV 2 - Rollbergesiedlung,11,Lübarser Straße,123022
445,12302212,488661.0,0.488661,283173.0,11239.0,0.023000,0.000000,0.942441,0.057559,0.000000,...,0.000000,12,Reinickendorf,30,Waidmannslust,22,MV 2 - Rollbergesiedlung,12,Rollbergesiedlung,123022
446,12304313,1843193.0,1.843193,492905.0,8118.0,0.004404,0.433026,0.000000,0.120229,0.284939,...,0.000000,12,Reinickendorf,30,Waidmannslust,43,West 3 - Borsigwalde/Freie Scholle,13,Borsigwalde,123043
447,12304314,2237979.0,2.237979,1044488.0,514576.0,0.229929,0.023517,0.000000,0.314247,0.007486,...,0.000000,12,Reinickendorf,30,Waidmannslust,43,West 3 - Borsigwalde/Freie Scholle,14,Ziekowstraße/Freie Scholle,123043


In [18]:
lor_info

,LOR,area,area_sqkm,living_area,leisure_area,leisure_part,altbau_flats,platte_flats,one_fam_flats,altbau_sqm,...,one_fam_pop,lower_quality,average_quality,upper_quality,Bezirk,Name,Prognoseraum,Progenoseraumname,Bezirksregion,Bezirksname
0,000000,890959877,890.959877,233329442,221881541,0.249037,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.322582,0.484049,0.193038,00,Berlin,00,Berlin,00,Berlin
1,010111,5165169,5.165169,124902,2002658,0.387724,0.710063,0.231318,0.000000,0.751963,...,0.000000,0.046380,0.756490,0.196129,01,Mitte,01,Zentrum,11,Tiergarten Süd
2,010112,2702649,2.702649,178220,27498,0.010174,0.136950,0.826960,0.000000,0.112249,...,0.000000,0.000000,0.124392,0.870742,01,Mitte,01,Zentrum,12,Regierungsviertel
3,010113,6228553,6.228553,1008603,246654,0.039601,0.472250,0.419428,0.000000,0.506993,...,0.000000,0.000363,0.655512,0.341050,01,Mitte,01,Zentrum,13,Alexanderplatz
4,010114,1755444,1.755444,620375,245531,0.139868,0.974208,0.000000,0.000000,0.948275,...,0.000000,0.000000,0.397393,0.602607,01,Mitte,01,Zentrum,14,Brunnenstraße Süd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,122311,13854910,13.854910,7343199,3462740,0.249929,0.009340,0.000000,0.946414,0.007510,...,0.943561,0.000000,0.256306,0.743694,12,Reinickendorf,23,Frohnau-Hermsdorf,11,Nord 1 - Frohnau/Hermsdorf
136,123012,11825409,11.825409,4098800,2625156,0.221993,0.032944,0.071991,0.494422,0.024776,...,0.529155,0.000412,0.999113,0.000000,12,Reinickendorf,30,Waidmannslust,12,Nord 2 - Waidmannslust/Wittenau/Lübars
137,123021,3146413,3.146413,1747281,123289,0.039184,0.000000,0.920163,0.079837,0.000000,...,0.085535,0.789693,0.210307,0.000000,12,Reinickendorf,30,Waidmannslust,21,MV 1 - Märkisches Viertel
138,123022,1618108,1.618108,748067,71730,0.044330,0.000000,0.573296,0.235994,0.000000,...,0.261701,0.947919,0.052081,0.000000,12,Reinickendorf,30,Waidmannslust,22,MV 2 - Rollbergesiedlung
